# Exercise 0: Can _hls4ml_ Convert the Full GTK Transformer?

## Objectives
In this exercise, you will:
1. Find out which layers _hls4ml_ can convert, for models in ONNX and in PyTorch
2. Check a small MLP and convert it with _hls4ml_
3. Check the trained GTK Transformer operation by operation, and try to convert it
4. Find which parts of the model block the conversion, and what we can do about it

## Instructions
Complete the code cells marked with `# TODO` comments. Follow the hints provided.

## Part 1: Environment Setup
Run the following cells to set up the environment (no changes needed). <br>
No Xilinx tool is needed for this exercise: start Jupyter after `HI_CONDA_ENV=hybrid-tutorial source env.sh py`.

In [ ]:
# Load libraries and import packages (no changes needed)

# General imports
import sys
from collections import Counter
from pathlib import Path

# Numpy and plotting
import numpy as np
import matplotlib.pyplot as plt

# ONNX and ONNX Runtime, to read and run the trained model
import onnx
import onnxruntime
from onnx import numpy_helper

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# hls4ml
import hls4ml

# Project folders: the project is the folder with hi_common.py, this exercise is in its school/ folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "hi_common.py").exists():
        PROJECT_DIR = folder
        break
SCHOOL_DIR = PROJECT_DIR / "school"
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(SCHOOL_DIR))

# Helpers of the project: configuration, events, trained checkpoint
import hi_common

# Local file with some useful methods (plots, printing)
from utils import *

# Configuration, trained model and output folders
CONFIG = hi_common.load_config(PROJECT_DIR / "config" / "hybrid_small_data.yaml")
TRAINED_MODEL = PROJECT_DIR / "artifacts" / "ckpt-e7c02308" / "onnx" / "full_current.onnx"
WORK_DIR = SCHOOL_DIR / "work"
FIGURE_DIR = WORK_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FPGA_PART = CONFIG["hls"]["part"]                   # FPGA part of the Alveo U55C - DO NOT CHANGE
model_config, _ = hi_common.model_config(CONFIG)

use_course_style()
print(f"hls4ml {hls4ml.__version__}, torch {torch.__version__}, onnx {onnx.__version__}")
print("Environment setup correctly!")

_hls4ml_ is a Python library that converts a trained neural network into FPGA firmware. It reads the model through a **frontend** (one per framework: PyTorch, ONNX, Keras), builds its own graph of layers, and a **backend** writes C++ for each layer, computing with *fixed-point* numbers instead of floats. Vitis HLS then turns the C++ into a hardware design, and Vivado into a bitstream for the FPGA.

For additional details:
- here is the official documentation: https://fastmachinelearning.org/hls4ml/index.html
- here is the GitHub page: https://github.com/fastmachinelearning/hls4ml

**Important:** each frontend can only read the layers it knows. A layer that is not on its list stops the conversion at step 2, and nothing else can happen.

In [ ]:
# Visualize the hls4ml tool chain (no changes needed)
plot_toolchain(save_to=FIGURE_DIR / "toolchain.png")
print("Plot saved as 'work/figures/toolchain.png'")

## Part 2: What Can _hls4ml_ Read?

### Task 2.1: The Supported Layers
_hls4ml_ keeps, for each frontend, the list of the layers it can convert. Get the two lists we need in this exercise: the one of the **ONNX** frontend and the one of the **PyTorch** frontend.

In [ ]:
# TODO: get the lists of the layers supported by the ONNX and the PyTorch frontends of hls4ml

# Hint: the two functions are in hls4ml.converters, and their names start with get_supported_
# Hint: in a notebook, type hls4ml.converters.get_ and press Tab to see the available names

onnx_layers = None     # Replace with correct code
pytorch_layers = None  # Replace with correct code

print(f"ONNX frontend   : {len(onnx_layers)} layers -> {sorted(onnx_layers)}")
print(f"PyTorch frontend: {len(pytorch_layers)} layers -> {sorted(pytorch_layers, key=str.lower)}")

### Task 2.2: Visualize the Supported Layers
The two lists, sorted by the kind of layer.

In [ ]:
# Visualize the layers supported by the two frontends (no changes needed)
plot_layer_catalogue(onnx_layers, pytorch_layers, save_to=FIGURE_DIR / "layer_catalogue.png")
print("Plot saved as 'work/figures/layer_catalogue.png'")

Look at the *Normalisation* column: the PyTorch frontend can read a layer normalisation (`LayerNorm`), the ONNX frontend cannot (there is no `LayerNormalization` in its list). <br>
**Important:** the frontends support different sets of layers, so the same network can be convertible from one format and not from the other. Always check a model against the list of the frontend you are going to use.

## Part 3: A First Model, a Small MLP
Before the real model, a small one: a multilayer perceptron (MLP) with 64 inputs, a hidden layer of 64 neurons with a ReLU activation, and 2 outputs. Its weights are random: for now we only care about its layers.

In [ ]:
# Define the small MLP (no changes needed)
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 64)    # 64 inputs -> 64 neurons
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)     # 64 neurons -> 2 outputs

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


mlp = SmallMLP().eval()
print(mlp)

### Task 3.1: Check the Layers of the MLP
The PyTorch frontend looks at every module (`torch.nn.Module`) of the model and checks its **class name** against its list. Do the same check by hand: find the modules of the MLP whose class name is not in `pytorch_layers`.

In [ ]:
# TODO: collect the class names of the MLP modules that are not supported by the PyTorch frontend

# Hint: start from an empty list, and add the class names that are not in pytorch_layers
# Hint: mlp.named_modules() gives (name, module) pairs; the first one is the model itself (empty name): skip it
# Hint: the class name of a module is type(module).__name__

mlp_unsupported = None  # Replace with correct code: the list of the unsupported class names

# YOUR CODE HERE

print(f"Unsupported modules: {mlp_unsupported}")

### Task 3.2: Convert the MLP with _hls4ml_
Now let _hls4ml_ do it. Converting a PyTorch model takes two steps:
- `hls_config`: a configuration created from the model, with the conversion options of every layer
   - here is where, later in the course, we will choose the precision and the parallelism of each layer
- the conversion itself, which uses the model, the `hls_config`, the FPGA part and some I/O options, and returns the _hls4ml_ model

In [ ]:
# TODO: create the hls4ml configuration of the MLP, then convert the MLP

# Hint: hls4ml.utils.config_from_pytorch_model(model, input_shape, granularity='name', backend='Vivado')
#       where input_shape is a list with one shape per input, without the batch dimension: here [(64,)]
# Hint: hls4ml.converters.convert_from_pytorch_model(model, hls_config=..., output_dir=..., backend='Vivado',
#                                                     io_type='io_parallel', part=FPGA_PART)
#       with output_dir=str(WORK_DIR / 'hls' / 'small_mlp')

mlp_config = None     # Replace with correct code
mlp_hls_model = None  # Replace with correct code

print("=" * 20, "HLS Config", "=" * 20)
print_dict(mlp_config)
print("=" * 50)

In [ ]:
# Inspect the layers of the hls4ml model (no changes needed)
show_hls_layers(mlp_hls_model)

The conversion works: every module of the MLP becomes an _hls4ml_ layer. Each layer has its **fixed-point** types, because the FPGA does not compute with floats: `fixed<16,6,TRN,WRAP,0>` (written `ap_fixed<16,6>` in the C++) is a number of 16 bits, 6 of them before the binary point, so it spans −32 to +32 in steps of 2⁻¹⁰ (`TRN`: extra bits are truncated, `WRAP`: a value out of range wraps around). These are the defaults for the inputs and the weights; the sums (`accum`) get wider types, so that nothing is lost.

### Bonus:
An _hls4ml_ model can do much more than this:
```python
hls_model.compile()             # build a C++ emulation of the firmware
y_hls = hls_model.predict(X)    # run it, with the same fixed-point arithmetic as the FPGA
hls_model.build(csim=False)     # run the synthesis: resources and latency
```
We will use them in the next exercises, on the model that matters.

## Part 4: The GTK Transformer in ONNX
Now the real model. For each event it reads the hits (x, y, GTK station, time) and gives, for every pair of hits *(i, j)* with *i ≤ j*, the probability that they are consecutive hits of the same particle. An event with *N* hits has *K = N(N+1)/2* pairs. Inside, two **Transformer** layers let every hit "look at" all the others (*attention*) before the pairs are scored.

The trained model is stored in **ONNX**, a standard format for neural networks: a graph of operations (`MatMul`, `Add`, `Softmax`, ...), where each node is one operation.

In [ ]:
# Load the trained model and inspect it (no changes needed)
model = onnx.load(TRAINED_MODEL)

for graph_input in model.graph.input:
    axes = [axis.dim_param or axis.dim_value for axis in graph_input.type.tensor_type.shape.dim]
    print(f"Input {graph_input.name}: {axes}")
print(f"{len(model.graph.node)} nodes; the first ten: {[node.op_type for node in model.graph.node[:10]]}")

### Task 4.1: Count the Operations
Every node of the graph has an operation type. Count how many nodes there are of each type, and find the types that the ONNX frontend of _hls4ml_ does not support.

In [ ]:
# TODO: count the nodes of each operation type, and list the operation types not supported by hls4ml

# Hint: the nodes are in model.graph.node, and the operation type of a node is node.op_type
# Hint: a collections.Counter counts them in one line
# Hint: compare the operation types with onnx_layers (hls4ml also accepts Dropout and Identity,
#       but this model has none)

operation_counts = None        # Replace with correct code: {operation type: number of nodes}
unsupported_operations = None  # Replace with correct code: list of the unsupported operation types

for operation, count in operation_counts.most_common():
    print(f"{operation:20s} {count:3d}  {'NOT SUPPORTED' if operation in unsupported_operations else 'supported'}")
print(f"\n{len(unsupported_operations)} unsupported operation types: {unsupported_operations}")

### Task 4.2: Visualize the Operations

In [ ]:
# Visualize the operations of the model (no changes needed)
plot_operation_counts(operation_counts, onnx_layers, save_to=FIGURE_DIR / "operation_counts.png")
print("Plot saved as 'work/figures/operation_counts.png'")

### Task 4.3: Try the Conversion
Lists are one thing: let's try the real conversion with the ONNX frontend. It will raise an exception: catch it and keep its message.

In [ ]:
# TODO: try to convert the ONNX model with hls4ml, and keep the error message

# Hint: hls4ml.converters.convert_from_onnx_model(model, output_dir=..., backend='Vivado',
#                                                 io_type='io_parallel', part=FPGA_PART)
#       with output_dir=str(WORK_DIR / 'hls' / 'gtk_onnx')
# Hint: the conversion raises an exception: the except block keeps its message

onnx_error = None
try:
    # YOUR CODE HERE
    pass
except Exception as error:
    onnx_error = str(error)

print(f"\nhls4ml says: {onnx_error}")

Look carefully at the output above:
- _hls4ml_ names only **one** operation, while in Task 4.1 you found six. The converter walks through the nodes in order and stops at the **first** one it cannot read (`Constant` is the very first node of the graph); it never looks at the rest. This is why it is worth checking the whole list first.
- the `Input shape` lines say `[4]` and `[2]`, but the input is `data [num_nodes, 4]`. _hls4ml_ takes the first axis of every input as the **batch** axis and drops it, so it would treat every hit as an independent sample. In this model the hits of an event are not independent: the attention mixes all of them. And `num_nodes` changes from event to event, while an FPGA design has fixed sizes.

### Task 4.4: Fold the Constants
The `Constant` nodes are only a by-product of the export: their values can be stored in the model as constant tensors (*initializers*), as ONNX clean-up tools do. The cell below does it, and checks with ONNX Runtime that the model still gives exactly the same output on a real event.

In [ ]:
# Replace the Constant nodes with constant tensors, and check the model output (no changes needed)
def fold_constants(onnx_model):
    folded = onnx.ModelProto()
    folded.CopyFrom(onnx_model)
    other_nodes = []
    for node in folded.graph.node:
        if node.op_type == "Constant":
            value = numpy_helper.to_array(node.attribute[0].t)
            folded.graph.initializer.append(numpy_helper.from_array(value, node.output[0]))
        else:
            other_nodes.append(node)
    del folded.graph.node[:]
    folded.graph.node.extend(other_nodes)
    return folded


folded_model = fold_constants(model)

event = hi_common.load_events(CONFIG, model_config, CONFIG["data"]["eval_split"], 1)[0]
inputs = {"data": event["features"], "pair_indices": hi_common.pair_index(len(event["features"]))}
before = onnxruntime.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"]).run(None, inputs)[0]
after = onnxruntime.InferenceSession(folded_model.SerializeToString(), providers=["CPUExecutionProvider"]).run(None, inputs)[0]
print(f"{len(model.graph.node)} -> {len(folded_model.graph.node)} nodes")
print(f"Same output on an event of {len(event['features'])} hits: {np.array_equal(before, after)}")

In [ ]:
# TODO: try again the conversion, this time on the folded model

# Hint: the same call as in Task 4.3, on folded_model, with output_dir=str(WORK_DIR / 'hls' / 'gtk_onnx_folded')

folded_error = None
try:
    # YOUR CODE HERE
    pass
except Exception as error:
    folded_error = str(error)

print(f"\nhls4ml says: {folded_error}")

Folding the constants only moved the problem to the next node. The other unsupported operations are not by-products of the export: they are part of what the model computes.

### Task 4.5: Where Are the Unsupported Operations?
The model has three parts:
- the **encoder**: an embedding and two Transformer layers, which turn every hit into 32 numbers (its *embedding*)
- the **pair building**: for every pair *(i, j)*, it picks the embeddings of hits *i* and *j* and puts them side by side (64 numbers)
- the **pair classifier**: a small network that scores each pair, then the final softmax

The plot shows every node of the graph, in order, grouped by part (`model_parts` in `utils.py` finds the parts from the graph itself).

In [ ]:
# Visualize where the unsupported operations are (no changes needed)
plot_where_unsupported(model, onnx_layers, save_to=FIGURE_DIR / "where_unsupported.png")
print("Plot saved as 'work/figures/where_unsupported.png'")

Every unsupported node is in the encoder (`Constant`, `Unsqueeze`, `Shape`, `Gather`, `Slice`, `LayerNormalization`) or in the pair building (`Gather` and two `Constant` nodes). The pair classifier uses only `MatMul`, `Add`, `Relu` and `Softmax`, all supported. Keep this in mind: it will be important at the end of the exercise.

## Part 5: The GTK Transformer in PyTorch
In Part 2 you saw that the PyTorch frontend knows more layers, `LayerNorm` among them. Maybe it can do better? The trained weights are also stored in the PyTorch checkpoint that `full_current.onnx` was exported from. The training code cannot be read by _hls4ml_ as it is (its `forward` takes pandas tables), so the cell below puts its **trained modules** in a plain `forward`, and checks that it gives the same probabilities as the ONNX model.

In [ ]:
# Load the trained network and check it against the ONNX model (no changes needed)
model_config, state_dict, checkpoint_info, trained_network, _ = hi_common.load_model(CONFIG)


class TrainedTransformer(nn.Module):
    def __init__(self, trained):
        super().__init__()
        self.embedding = trained.embedding                      # Linear 4 -> 32
        self.transformer_encoder = trained.transformer_encoder  # 2 Transformer encoder layers
        self.fc1 = trained.fc1                                  # pair classifier: Linear 64 -> 64
        self.fc2 = trained.fc2                                  # Linear 64 -> 2

    def forward(self, hits, pair_indices):
        x = self.embedding(hits.unsqueeze(0))                                            # [1, N, 32]
        x = self.transformer_encoder(x)                                                  # [1, N, 32]
        pairs = torch.cat([x[:, pair_indices[:, 0]], x[:, pair_indices[:, 1]]], dim=-1)  # [1, K, 64]
        logits = self.fc2(torch.relu(self.fc1(pairs)))                                   # [1, K, 2]
        return torch.softmax(logits, dim=-1)


trained_model = TrainedTransformer(trained_network).eval()

onnx_session = onnxruntime.InferenceSession(str(TRAINED_MODEL), providers=["CPUExecutionProvider"])
largest_difference = 0.0
for test_event in hi_common.load_events(CONFIG, model_config, CONFIG["data"]["eval_split"], 20):
    hits = test_event["features"]
    pairs = hi_common.pair_index(len(hits))
    with torch.no_grad():
        torch_probabilities = trained_model(torch.from_numpy(hits), torch.from_numpy(pairs)).numpy()
    onnx_probabilities = onnx_session.run(None, {"data": hits, "pair_indices": pairs})[0]
    largest_difference = max(largest_difference, float(np.abs(torch_probabilities - onnx_probabilities).max()))
print(f"PyTorch vs ONNX on 20 events: largest difference in probability {largest_difference:.1e} (the same model)")

### Task 5.1: Check the Modules
As in Task 3.1, find the modules of `trained_model` whose class name the PyTorch frontend does not support. Printing the name and the class of every module also shows you the structure of the model.

In [ ]:
# TODO: collect the class names of the modules of trained_model that are not supported (each name only once)

# Hint: as in Task 3.1, loop over trained_model.named_modules() and skip the model itself
# Hint: skip also ModuleList (only a container: it is never called) and Dropout (hls4ml removes it:
#       it does nothing at inference)

unsupported_modules = None  # Replace with correct code: the list of the unsupported class names

# YOUR CODE HERE

print(f"\nUnsupported modules: {unsupported_modules}")

`NonDynamicallyQuantizableLinear` is the output projection inside the attention: a `Linear` under another class name, so _hls4ml_ does not recognise it. The real obstacle is the attention itself, as the next tasks show.

### Task 5.2: Try the Conversion
Ask _hls4ml_ for the configuration of `trained_model`. _hls4ml_ needs fixed input sizes: use those of a median event, 51 hits and 1,326 pairs.

In [ ]:
# Input shapes of a median event (no changes needed)
MEDIAN_EVENT_SHAPES = [(51, 4), (51 * 52 // 2, 2)]       # hits [N, 4] and pair_indices [K, 2]

In [ ]:
# TODO: try to create the hls4ml configuration of trained_model, and keep the error message

# Hint: hls4ml.utils.config_from_pytorch_model(model, MEDIAN_EVENT_SHAPES, granularity='name', backend='Vivado')

pytorch_error = None
try:
    # YOUR CODE HERE
    pass
except Exception as error:
    pytorch_error = str(error)

print(f"hls4ml says: {pytorch_error}")

### Bonus:
_hls4ml_ sees `transformer_encoder` as one layer and does not know it. What if we open it, and call the modules of each Transformer layer one by one, as `nn.TransformerEncoderLayer` does internally (attention, residual, `LayerNorm`, feed-forward, residual, `LayerNorm`)?

In [ ]:
# Call the modules of each Transformer layer one by one, and try again (no changes needed)
class UnpackedTransformer(nn.Module):
    def __init__(self, trained):
        super().__init__()
        self.embedding = trained.embedding
        self.layers = trained.transformer_encoder.layers
        self.fc1 = trained.fc1
        self.fc2 = trained.fc2

    def forward(self, hits, pair_indices):
        x = self.embedding(hits.unsqueeze(0))
        for layer in self.layers:
            attention = layer.self_attn(x, x, x, need_weights=False)[0]       # MultiheadAttention
            x = layer.norm1(x + attention)
            x = layer.norm2(x + layer.linear2(F.relu(layer.linear1(x))))
        pairs = torch.cat([x[:, pair_indices[:, 0]], x[:, pair_indices[:, 1]]], dim=-1)
        return torch.softmax(self.fc2(torch.relu(self.fc1(pairs))), dim=-1)


try:
    hls4ml.utils.config_from_pytorch_model(UnpackedTransformer(trained_network).eval(), MEDIAN_EVENT_SHAPES,
                                           granularity="name", backend="Vivado")
except Exception as error:
    print(f"hls4ml says: {error}")

Now the `LayerNorm` modules would be fine, but the **attention** itself, the heart of a Transformer, is not supported. Neither frontend can convert our model as it is.

---

## So, is this the end of the tutorial? Does the FPGA journey stop here?

## No, all is not lost...

Look again at the plot of Task 4.5: **every** unsupported operation is in the encoder or in the pair building. The **pair classifier** uses only operations _hls4ml_ supports. So we do not need _hls4ml_ to convert the whole model: we can **split it**, keep the encoder on the CPU, and send the pair classifier to the FPGA.

## Part 6: The Pair Classifier
The cell below takes the trained weights of the pair classifier from the checkpoint.

In [ ]:
# Load the trained pair classifier (no changes needed)
trained_classifier = SmallMLP().eval()
trained_classifier.fc1.load_state_dict(trained_network.fc1.state_dict())
trained_classifier.fc2.load_state_dict(trained_network.fc2.state_dict())
print(trained_classifier)

### Task 6.1: Convert the Pair Classifier
Recognise it? It is the `SmallMLP` of Part 3, now with its **trained** weights: 64 inputs (the two 32-number embeddings of a pair) and 2 outputs (the two scores, "edge" and "no edge"). Convert it with _hls4ml_.

In [ ]:
# TODO: create the hls4ml configuration of trained_classifier, then convert it

# Hint: exactly as in Task 3.2, with output_dir=str(WORK_DIR / 'hls' / 'pair_classifier')

classifier_config = None     # Replace with correct code
classifier_hls_model = None  # Replace with correct code

show_hls_layers(classifier_hls_model)

The trained pair classifier converts without problems: the part of the model that _hls4ml_ supports is exactly the one you converted in Part 3.

### Task 6.2: Where Is the Work?
Is the pair classifier worth the effort? The encoder runs once per **hit**, the pair classifier once per **pair**: an event with *N* hits has *K = N(N+1)/2* pairs. The plot counts the multiply-adds of each part (the basic operation of a neural network) for the events of the test set.

In [ ]:
# Visualize where the work is (no changes needed)
hits_per_event = load_hits_per_event(CONFIG, model_config)
classifier_share = plot_work_share(hits_per_event, save_to=FIGURE_DIR / "work_share.png")
print("Plot saved as 'work/figures/work_share.png'")

The pair classifier runs once per pair, and the number of pairs grows like *N²*; the work of the encoder grows mostly like *N* (only its attention grows like *N²*, and it is cheap compared with the 4,224 multiply-adds of the classifier on every pair). For a median event of 51 hits, about 83% of the work is in the pair classifier (84% over the whole test set), and its share grows with the number of hits.

### Task 6.3: The Plan
This is what the next exercises do:
- the **encoder** stays on the CPU, where ONNX Runtime runs it as it is: in Exercise 1 you will split it off the full model
- the **pair classifier** goes to the FPGA through _hls4ml_: this is the core of the course (Exercise 2)
- the **pair building**, which _hls4ml_ cannot do, also runs on the FPGA, in a small custom C++ kernel around the _hls4ml_ code, so that the card only receives the hit embeddings

**Important:** the custom C++ kernel for the pair building is **already written for you**: it is too complex for this course, and you will not write it. You will focus on the conversion with _hls4ml_; we will only show you how a custom kernel is connected to an _hls4ml_ network.

In [ ]:
# Visualize the plan (no changes needed)
plot_plan(save_to=FIGURE_DIR / "plan.png")
print("Plot saved as 'work/figures/plan.png'")

| exercise | what you will do |
|---|---|
| **1**: the ONNX graph and the split | analyse the graph, find where to cut it, split the model into `encoder.onnx` and `pair_head.onnx`, check that the two parts give the same result |
| **2**: _hls4ml_ and the firmware | convert the pair classifier with _hls4ml_, choose its fixed-point precision, estimate the FPGA resources; see how the provided pair-building kernel is connected to the _hls4ml_ network to make the firmware |
| **3**: inference on the FPGA | run the hybrid model on the Alveo U55C and compare it with the CPU |